# Ler arquivos .h5 e mostrar shapes
Este notebook procura por arquivos `*.h5` na pasta atual, abre cada arquivo com `h5py` e imprime os shapes (e dtypes) de todos os datasets encontrados.

Observação: por padrão o diretório inspecionado é o diretório de trabalho atual do notebook; você pode alterar a variável `DATA_DIR` na célula de código se necessário.

In [2]:
import h5py
from pathlib import Path
import sys

# Diretório com os arquivos .h5 (mude aqui se quiser)
DATA_DIR = Path('.')

h5_files = sorted(DATA_DIR.glob('*.h5'))
print(f'Procurando arquivos .h5 em: {DATA_DIR.resolve()}')
if not h5_files:
    print('Nenhum arquivo .h5 encontrado.')

for p in h5_files:
    print('\nArquivo:', p.name)
    try:
        with h5py.File(p, 'r') as f:
            def print_datasets(name, obj):
                if isinstance(obj, h5py.Dataset):
                    print(f'  {name} -> shape: {obj.shape}, dtype: {obj.dtype}')
            f.visititems(print_datasets)
    except Exception as e:
        print('  Erro ao ler:', e)

Procurando arquivos .h5 em: C:\Users\mateu\clones\lvl\erros

Arquivo: model_compound_4_10_34_1_interface.h5
  N_compound -> shape: (101, 1, 5, 1, 16, 2000), dtype: float64
  N_compound_maxenv -> shape: (101, 1, 5, 1, 16), dtype: float64

Arquivo: model_compound_4_10_68_1_interface.h5
  N_compound -> shape: (101, 1, 1, 1, 16, 2000), dtype: float64
  N_compound_maxenv -> shape: (101, 1, 1, 1, 16), dtype: float64


In [4]:
from pathlib import Path
import h5py
import numpy as np

# Arquivo alvo (modifique se precisar)
DATA_DIR = Path('.')
file_path = DATA_DIR / 'model_compound_4_10_34_1_interface.h5'
print(f'Arquivo alvo: {file_path.resolve()}')
if not file_path.exists():
    print('Arquivo não encontrado:', file_path)
else:
    with h5py.File(file_path, 'r') as f:
        found = {'v': False}
        def check(name, obj):
            # procura datasets cuja primeira dimensão tem tamanho 101
            if isinstance(obj, h5py.Dataset):
                shape = getattr(obj, 'shape', None)
                if shape and len(shape) >= 1 and shape[0] == 101:
                    found['v'] = True
                    print('\nDataset: {} shape: {}, dtype: {}'.format(name, shape, obj.dtype))
                    # ler dados (atenção: pode usar muita memória para datasets grandes)
                    data = obj[()]
                    # valores distintos na primeira dimensão
                    try:
                        if getattr(data, 'ndim', 0) == 1:
                            uniques0 = np.unique(data)
                        else:
                            uniques0 = np.unique(data, axis=0)
                    except Exception:
                        uniques0 = np.unique(np.asanyarray(data).ravel())
                    print('Valores distintos na dimensão 0 (count={}):'.format(getattr(uniques0, 'shape', [len(uniques0)])[0]))
                    print(uniques0)
                    # para cada outra dimensão, mostra índices amostrados e sample dos valores nessa fatia
                    ndim = len(shape)
                    for axis in range(1, ndim):
                        axis_len = shape[axis]
                        print(f'\nDim {axis} (size={axis_len}):')
                        # escolhe até 5 índices amostrados uniformemente
                        sample_indices = np.unique(np.linspace(0, axis_len-1, min(5, axis_len), dtype=int))
                        for idx in sample_indices:
                            # cria slice para selecionar a fatia em axis=idx
                            slicer = [slice(None)] * ndim
                            slicer[axis] = int(idx)
                            try:
                                vals = data[tuple(slicer)]
                            except Exception:
                                vals = obj[tuple(slicer)]
                            vals = np.asanyarray(vals).ravel()
                            try:
                                uniques = np.unique(vals)
                            except Exception:
                                uniques = np.unique(vals.astype(str))
                            n_show = 20
                            if getattr(uniques, 'size', 1) > n_show:
                                sample = uniques[:n_show]
                                more = f'... (and {uniques.size - n_show} more)'
                            else:
                                sample = uniques
                                more = ''
                            print(f'  index {idx}: distinct count={getattr(uniques, "size", len(uniques))} sample={sample} {more}')
        f.visititems(check)
        if not found['v']:
            print('Nenhum dataset com primeira dimensão igual a 101 foi encontrado.')

Arquivo alvo: C:\Users\mateu\clones\lvl\erros\model_compound_4_10_34_1_interface.h5

Dataset: N_compound shape: (101, 1, 5, 1, 16, 2000), dtype: float64
Valores distintos na dimensão 0 (count=101):
[[[[[[-1.64778291e-05 -1.50006385e-05 -1.32328578e-05 ...
      -1.87462641e-05 -1.83851086e-05 -1.76188420e-05]
     [-6.79499516e-06 -6.13395476e-06 -5.30599648e-06 ...
      -7.68651991e-06 -7.57547852e-06 -7.27770699e-06]
     [ 3.91700578e-07  3.93176030e-07  3.94468422e-07 ...
       3.86282168e-07  3.88224800e-07  3.90043392e-07]
     ...
     [ 4.31907637e-07  4.29538201e-07  4.26991320e-07 ...
       4.38069587e-07  4.36143148e-07  4.34102078e-07]
     [ 1.46293144e-06  1.44539663e-06  1.42766038e-06 ...
       1.51459255e-06  1.49743826e-06  1.48025907e-06]
     [ 3.26344171e-06  5.55498968e-06  7.62971259e-06 ...
      -4.39500131e-06 -1.76868656e-06  8.03956829e-07]]]


   [[[ 1.13141257e-05  1.05517808e-05  9.55376989e-06 ...
       1.19295895e-05  1.20228248e-05  1.18121480e-05

In [6]:
from pathlib import Path
import h5py
import numpy as np

# Ajuste se necessário
DATA_DIR = Path('.')
file_path = DATA_DIR / 'model_compound_4_10_34_1_interface.h5'
print(f"Arquivo alvo (para N_compound_maxenv): {file_path.resolve()}")

if not file_path.exists():
    print('Arquivo não encontrado:', file_path)
else:
    with h5py.File(file_path, 'r') as f:
        # tenta acesso direto
        dset = f.get('N_compound_maxenv')
        # se não achar, procura por qualquer dataset com esse sufixo
        if dset is None:
            found_path = None
            def finder(name, obj):
                if isinstance(obj, h5py.Dataset) and name.endswith('N_compound_maxenv'):
                    found_path = name
            f.visititems(finder)
            if found_path:
                dset = f[found_path]
                print(f"Dataset encontrado em: {found_path}")

        if dset is None:
            print("Dataset 'N_compound_maxenv' não encontrado no arquivo.")
        else:
            print('Dataset shape:', getattr(dset, 'shape', None))
            try:
                data = dset[()]
            except Exception as e:
                print('Erro ao ler dados do dataset:', e)
                data = None

            if data is None:
                pass
            else:
                arr = np.asanyarray(data)
                ndim = arr.ndim
                total = arr.size
                print(f'ndim={ndim}, total_elements={total}')

                # mostra o valor no índice zero para todos os eixos (ex: data[0,0,...])
                try:
                    zeros_idx = tuple(0 for _ in range(ndim))
                    print('\nValor em todos índices zero (posição {}):'.format(zeros_idx))
                    print(arr[zeros_idx])
                except Exception as e:
                    print('Não foi possível indexar todos zeros:', e)

                # para cada eixo, imprime a fatia onde indice=0
                for axis in range(ndim):
                    slicer = [slice(None)] * ndim
                    slicer[axis] = 0
                    try:
                        vals = arr[tuple(slicer)]
                    except Exception:
                        # fallback para ler via dataset diretamente
                        try:
                            vals = dset[tuple(slicer)]
                        except Exception as ee:
                            print(f'  Erro ao extrair slice axis={axis} index=0: {ee}')
                            continue
                    vals = np.asanyarray(vals)
                    print(f"\nSlice axis {axis} index 0 -> shape={vals.shape}")
                    # se pequeno, mostra tudo; caso contrário mostra resumo
                    if vals.size <= 2000:
                        print(vals)
                    else:
                        flat = vals.ravel()
                        print('  Primeiro 20 valores da slice:', flat[:20])
                        print(f'  ... (e mais {flat.size-20} valores)')


Arquivo alvo (para N_compound_maxenv): C:\Users\mateu\clones\lvl\erros\model_compound_4_10_34_1_interface.h5
Dataset shape: (101, 1, 5, 1, 16)
ndim=5, total_elements=8080

Valor em todos índices zero (posição (0, 0, 0, 0, 0)):
0.0022279471426718753

Slice axis 0 index 0 -> shape=(1, 5, 1, 16)
[[[[0.00222795 0.00206735 0.00164373 0.00110438 0.00206656 0.00222774
    0.00206649 0.00164264 0.00164321 0.00206656 0.00222772 0.00206672
    0.00110434 0.00164322 0.00206654 0.00222782]]

  [[0.00223216 0.00207179 0.0016503  0.00111231 0.00207204 0.00223249
    0.00207244 0.00165133 0.00165017 0.00207204 0.00223247 0.00207236
    0.00111225 0.00165018 0.00207202 0.00223223]]

  [[0.00223736 0.00207813 0.00165782 0.00112022 0.00207746 0.00223724
    0.00207759 0.00165685 0.00165708 0.00207746 0.00223721 0.0020775
    0.00112014 0.00165709 0.00207744 0.0022372 ]]

  [[0.00224177 0.00208282 0.00166392 0.00112814 0.00208291 0.00224195
    0.00208322 0.00166488 0.001664   0.00208291 0.00224192 0.002

## Como executar (PowerShell)
1) Instale dependências (se necessário):
```powershell
pip install h5py
```
2) Inicie o Jupyter e abra este notebook:
```powershell
jupyter notebook
```
Dica: se quiser apontar para outra pasta, altere a variável `DATA_DIR` na célula de código para o caminho desejado, por exemplo:
```python
from pathlib import Path
DATA_DIR = Path(r'c:\Users\mateu\clones\lvl\erros')
```